# Model Analysis

Step-by-step exploration of the production model — what it learned, what drives its decisions, and where it struggles.

**Contents**
1. Load the production artifact
2. Load feature data
3. SHAP — global feature importance
4. SHAP — per-class importance (BUY vs SELL vs HOLD)
5. *(next: per-ticker deep-dives, fold-by-fold performance, calibration)*

---
## 1. Setup

In [ ]:
import json
import pickle
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap

warnings.filterwarnings("ignore")
shap.initjs()

# Paths
REPO = Path("..")  # notebook lives in notebooks/
PRODUCTION_DIR = REPO / "models" / "production"
FEATURES_DIR   = REPO / "data" / "features"

# Signal labels
CLASS_NAMES = ["SELL", "HOLD", "BUY"]   # matches TARGET_ENCODING: 0, 1, 2

---
## 2. Load the production model

The artifact stores the fitted sklearn/LightGBM model, the exact feature list it was trained on, and the training config.

In [ ]:
# Find the most recently modified production artifact
artifact_dirs = sorted(
    [p for p in PRODUCTION_DIR.iterdir() if p.is_dir() and (p / "metadata.json").exists()],
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
assert artifact_dirs, "No production artifact found. Run `make train` first."

artifact_dir = artifact_dirs[0]
print(f"Artifact: {artifact_dir.name}")

In [ ]:
# Load model
with (artifact_dir / "model.pkl").open("rb") as f:
    artifact = pickle.load(f)

model    = artifact["model"]
features = artifact["features"]   # ordered list — same order used at fit time

print(f"Model type : {type(model).__name__}")
print(f"Features   : {len(features)}")
print(f"Classes    : {CLASS_NAMES}")

In [ ]:
# Show the production fold metrics
metadata = json.loads((artifact_dir / "metadata.json").read_text())
prod_fold = metadata.get("production_fold", {})

print("=== Production fold metrics ===")
for k, v in prod_fold.items():
    if isinstance(v, float):
        print(f"  {k:<30} {v:.4f}")
    elif not isinstance(v, dict):
        print(f"  {k:<30} {v}")

---
## 3. Load feature data

We'll use the most recent feature file per ticker — the same rows the server reads at inference time.  
NaN values are filled with 0.0, matching the imputation used during training.

In [ ]:
# Load the latest CSV for every ticker
frames = []
for ticker_dir in sorted(FEATURES_DIR.iterdir()):
    if not ticker_dir.is_dir() or ticker_dir.name == "runs":
        continue
    csvs = sorted(ticker_dir.glob("*.csv"))
    if csvs:
        frames.append(pd.read_csv(csvs[-1]))

df_raw = pd.concat(frames, ignore_index=True)
print(f"Loaded {len(frames)} tickers, {len(df_raw):,} rows total")
print(f"Date range: {df_raw['date'].min()}  →  {df_raw['date'].max()}")

In [ ]:
# Prepare the feature matrix — same pre-processing as production inference
X = df_raw[features].astype(float).fillna(0.0)

# Keep the labelled rows for any supervised analysis later
labelled_mask = df_raw["target"].notna()
X_labelled    = X[labelled_mask].reset_index(drop=True)
y_labelled    = df_raw.loc[labelled_mask, "target"].reset_index(drop=True)
tickers       = df_raw.loc[labelled_mask, "ticker"].reset_index(drop=True)

print(f"Feature matrix  : {X.shape}")
print(f"Labelled rows   : {X_labelled.shape[0]}")
print(f"Target dist:\n{y_labelled.value_counts()}")

---
## 4. SHAP — global feature importance

**What SHAP values tell us:** for each row and each feature, the SHAP value is the contribution of that feature to pushing the prediction away from the base rate.  
Positive → pushes toward this class. Negative → pushes away.

We use `TreeExplainer`, which is exact and fast for tree-based models.

In [ ]:
# Initialise TreeExplainer — may take ~10s for large forests
explainer = shap.TreeExplainer(model)
print("TreeExplainer ready.")

In [ ]:
# Compute SHAP values on labelled rows
# Result shape for multiclass: (n_samples, n_features, n_classes)
shap_values = explainer.shap_values(X_labelled)  # ndarray or list depending on SHAP version

# Normalise to a single 3-D array (n_samples, n_features, n_classes)
if isinstance(shap_values, list):
    # Older SHAP: list of (n_samples, n_features) per class
    shap_3d = np.stack(shap_values, axis=2)   # → (n_samples, n_features, n_classes)
else:
    shap_3d = shap_values                      # already (n_samples, n_features, n_classes)

print(f"SHAP values shape: {shap_3d.shape}")
print(f"  axis 0 = samples ({shap_3d.shape[0]})")
print(f"  axis 1 = features ({shap_3d.shape[1]})")
print(f"  axis 2 = classes  ({shap_3d.shape[2]}) → {CLASS_NAMES}")

In [ ]:
# Mean absolute SHAP across all classes → one importance score per feature
mean_abs_shap = np.abs(shap_3d).mean(axis=(0, 2))  # (n_features,)

importance_df = (
    pd.DataFrame({"feature": features, "mean_abs_shap": mean_abs_shap})
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)
importance_df.head(15)

In [ ]:
# ── Bar chart: overall feature importance ─────────────────────────────────────
top_n = 20
top = importance_df.head(top_n)

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(top["feature"][::-1], top["mean_abs_shap"][::-1], color="steelblue")
ax.set_xlabel("Mean |SHAP value| (averaged across all classes and samples)", fontsize=11)
ax.set_title(f"Top {top_n} most important features — {type(model).__name__}", fontsize=13, pad=12)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

---
## 5. SHAP — per-class beeswarm plots

The bar chart above shows *how much* each feature matters overall.  
The beeswarm plots below show *how* each feature pushes toward a specific signal.

Each dot is one sample. **Colour = feature value** (red = high, blue = low).  
**X-axis = SHAP value**: positive means the feature pushed the model toward this class.

In [ ]:
# Helper — build a shap.Explanation object for one class
def make_explanation(class_idx: int) -> shap.Explanation:
    return shap.Explanation(
        values     = shap_3d[:, :, class_idx],
        base_values= explainer.expected_value[class_idx] if hasattr(explainer.expected_value, "__len__") else explainer.expected_value,
        data       = X_labelled.values,
        feature_names = features,
    )

In [ ]:
# ── BUY class beeswarm ────────────────────────────────────────────────────────
print("BUY class — what features push the model toward a BUY signal?")
shap.plots.beeswarm(make_explanation(2), max_display=15, show=True)

In [ ]:
# ── SELL class beeswarm ───────────────────────────────────────────────────────
print("SELL class — what features push the model toward a SELL signal?")
shap.plots.beeswarm(make_explanation(0), max_display=15, show=True)

In [ ]:
# ── HOLD class beeswarm ───────────────────────────────────────────────────────
print("HOLD class — what features keep the model neutral?")
shap.plots.beeswarm(make_explanation(1), max_display=15, show=True)

---
## 6. Feature importance: signal features vs sentiment features

A quick audit to check whether the 7 sentiment features contribute meaningfully  
vs the 25 price/volume/calendar features.

In [ ]:
SENTIMENT_FEATURES = [
    "bullish_percent", "bearish_percent", "company_news_score",
    "article_count", "positive_insights", "negative_insights",
    "neutral_insights", "sentiment_available",
]

imp = importance_df.copy()
imp["group"] = imp["feature"].apply(
    lambda f: "sentiment" if f in SENTIMENT_FEATURES else "price/volume/calendar"
)

group_summary = (
    imp.groupby("group")["mean_abs_shap"]
    .agg(["mean", "sum", "count"])
    .rename(columns={"mean": "avg importance", "sum": "total importance", "count": "# features"})
    .sort_values("total importance", ascending=False)
)
display(group_summary.round(5))

print("\nSentiment features individually:")
display(imp[imp["group"] == "sentiment"][["feature", "mean_abs_shap"]].reset_index(drop=True).round(5))

---
*Next steps to add here:*
- **Section 7** — Fold-by-fold F1 and signal distribution over time
- **Section 8** — Per-ticker signal breakdown
- **Section 9** — Calibration: do confidence scores match actual accuracy?
- **Section 10** — SHAP waterfall for individual predictions